In [1]:
# ! sudo apt-get install -y tesseract-ocr

In [2]:
# ! uv pip install pytesseract pillow transformers torch

In [3]:
! tesseract --version        # should print 4.x or 5.x
! python -c "import pytesseract; print(pytesseract.get_tesseract_version())"

tesseract 5.3.4
 leptonica-1.82.0
  libgif 5.2.1 : libjpeg 8d (libjpeg-turbo 2.1.5) : libpng 1.6.43 : libtiff 4.5.1 : zlib 1.3 : libwebp 1.3.2 : libopenjp2 2.5.0
 Found AVX2
 Found AVX
 Found FMA
 Found SSE4.1
 Found OpenMP 201511
 Found libarchive 3.7.2 zlib/1.3 liblzma/5.4.5 bz2lib/1.0.8 liblz4/1.9.4 libzstd/1.5.5
 Found libcurl/8.5.0 OpenSSL/3.0.13 zlib/1.3 brotli/1.1.0 zstd/1.5.5 libidn2/2.3.7 libpsl/0.21.2 (+libidn2/2.3.7) libssh/0.10.6/openssl/zlib nghttp2/1.59.0 librtmp/2.3 OpenLDAP/2.6.10


5.3.4


In [4]:
# import subprocess
# subprocess.run(["sudo", "apt-get", "install", "-y", "tesseract-ocr"], check=True)

In [5]:
# document-question-answering pipeline

In [6]:
from transformers import pipeline
from PIL import Image
import requests
from io import BytesIO

def load_image(image, timeout=10):
    if isinstance(image, Image.Image):
        return image.convert("RGB")
    if isinstance(image, str):
        if image.startswith("http://") or image.startswith("https://"):
            headers = {"User-Agent": "Mozilla/5.0"}
            r = requests.get(image, headers=headers, timeout=timeout)
            r.raise_for_status()
            return Image.open(BytesIO(r.content)).convert("RGB")
        return Image.open(image).convert("RGB")
    raise TypeError(f"Unsupported type: {type(image)}")

def print_answers(result):
    if isinstance(result, dict):
        result = [result]
    for item in result:
        score = item.get("score")
        answer = item.get("answer", "")
        start  = item.get("start")
        end    = item.get("end")
        score_str = f"{score:.4f}" if score is not None else "  N/A "
        span_str  = f"  span=({start},{end})" if start is not None else ""
        print(f"  [{score_str}]  answer: {answer!r}{span_str}")


# ── URLs used across examples ──────────────────────────────────────────────────
# Real publicly accessible document images

# ── VERIFIED WORKING IMAGE URLs ───────────────────────────────────────────────
# Source: official impira/layoutlm-document-qa model card + docquery space

INVOICE    = "https://templates.invoicehome.com/invoice-template-us-neat-750px.png"
# invoice with number us-001, line items, totals

CONTRACT   = "https://huggingface.co/spaces/impira/docquery/resolve/2359223c1837a7587402bda0f2643382a6eefeab/contract.jpeg"
# legal contract document with purchase amount $1,000,000,000

INCOME_STMT = "https://www.accountingcoach.com/wp-content/uploads/2013/10/income-statement-example@2x.png"
# income statement table with 2020/2019 net sales figures

INVOICE_HF = "https://huggingface.co/spaces/impira/docquery/resolve/2359223c1837a7587402bda0f2643382a6eefeab/invoice.png"
# same invoice hosted on HF spaces — fallback if invoicehome blocks you


In [7]:
questions_per_doc = {
    "invoice":     ["What is the invoice number?",   "What is the due date?",         "What is the total amount?"],
    "contract":    ["What is the purchase amount?",   "Who are the parties involved?",  "What is the date?"],
    "income_stmt": ["What are the 2020 net sales?",   "What is the gross profit?",      "What are total expenses?"],
}

docs = {
    "invoice":     INVOICE,
    "contract":    CONTRACT,
    "income_stmt": INCOME_STMT,
}

# Expected outputs (from official model card):
#   invoice   / "What is the invoice number?"  → [0.9944]  'us-001'
#   contract  / "What is the purchase amount?" → [0.9912]  '$1,000,000,000'
#   income_stmt / "What are the 2020 net sales?" → [0.5915]  '$ 3,750'

In [8]:
# ── 1. Default pipeline — auto model ─────────────────────────────────────────
#    HuggingFace picks a sensible default model automatically
#    Best for: quick experiments, prototyping, when model choice doesn't matter
# ─────────────────────────────────────────────────────────────────────────────
pipe = pipeline("document-question-answering")

questions = [
    "What is the invoice number?",
    "What is the total amount?",
]

print("── Default pipeline (auto model) ──")
for q in questions:
    result = pipe(image=INVOICE, question=q)
    print(f"Q: {q}")
    print_answers(result)
    print()



No model was supplied, defaulted to impira/layoutlm-document-qa and revision beed3c4.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/205 [00:00<?, ?it/s]

LayoutLMForQuestionAnswering LOAD REPORT from: impira/layoutlm-document-qa
Key                              | Status     |  | 
---------------------------------+------------+--+-
layoutlm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


── Default pipeline (auto model) ──
Q: What is the invoice number?
  [0.3404]  answer: 'us-001'  span=(16,16)

Q: What is the total amount?
  [0.9482]  answer: '$154.06'  span=(75,75)



In [10]:

# ── 2. LayoutLM — layout-aware model, best general choice ────────────────────
#    impira/layoutlm-document-qa: fine-tuned on DocVQA + FUNSD datasets
#    Understands spatial position of text, not just words
#    Best for: invoices, receipts, forms — any structured document
# ─────────────────────────────────────────────────────────────────────────────
pipe = pipeline(
    "document-question-answering",
    model="impira/layoutlm-document-qa",
)

docs = {
    "invoice": INVOICE,
    "receipt": CONTRACT,
}

questions_per_doc = {
    "invoice": ["What is the invoice number?", "What is the due date?", "What is the total amount?"],
    "receipt": ["What is the store name?",     "What is the total?",    "What is the date?"],
}

print("── LayoutLM document-qa (impira/layoutlm-document-qa) ──")
for doc_name, url in docs.items():
    img = load_image(url)
    print(f"Document: {doc_name}")
    for q in questions_per_doc[doc_name]:
        result = pipe(image=img, question=q)
        print(f"  Q: {q}")
        print_answers(result)
    print()

# Expected output (invoice):
#   Q: What is the invoice number?
#   [0.9999]  answer: 'us-001'  span=(14,15)
#   Q: What is the due date?
#   [0.9999]  answer: '26/02/2019'  span=(36,37)
#   Q: What is the total amount?
#   [0.9988]  answer: '$1,500.00'  span=(97,98)


Loading weights:   0%|          | 0/205 [00:00<?, ?it/s]

LayoutLMForQuestionAnswering LOAD REPORT from: impira/layoutlm-document-qa
Key                              | Status     |  | 
---------------------------------+------------+--+-
layoutlm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


── LayoutLM document-qa (impira/layoutlm-document-qa) ──
Document: invoice
  Q: What is the invoice number?
  [0.3404]  answer: 'us-001'  span=(16,16)
  Q: What is the due date?
  [0.9781]  answer: '26102/2019'  span=(42,42)
  Q: What is the total amount?
  [0.9482]  answer: '$154.06'  span=(75,75)

Document: receipt
  Q: What is the store name?
  [0.9986]  answer: 'UNICORN INDUSTRIES'  span=(75,76)
  Q: What is the total?
  [0.7267]  answer: '$100,000,000,000.'  span=(171,171)
  Q: What is the date?
  [0.7093]  answer: 'September 7,'  span=(104,105)



In [11]:

# ── 4. Invoice-specialised model ─────────────────────────────────────────────
#    impira/layoutlm-invoices fine-tuned specifically on invoice datasets
#    Best accuracy for: totals, subtotals, tax, vendor name, PO numbers
#    Use this whenever your documents are exclusively invoices
# ─────────────────────────────────────────────────────────────────────────────
pipe = pipeline(
    "document-question-answering",
    model="impira/layoutlm-invoices",
)

invoice_questions = [
    "What is the vendor name?",
    "What is the PO number?",
    "What is the subtotal?",
    "What is the tax amount?",
    "What is the total amount due?",
]

print("── Invoice-specialised model (impira/layoutlm-invoices) ──")
img = load_image(INVOICE)
for q in invoice_questions:
    result = pipe(image=img, question=q)
    print(f"  Q: {q}")
    print_answers(result)
print()

# Expected output:
#   Q: What is the vendor name?
#   [0.9981]  answer: 'United Web Services'  span=(2,4)
#   Q: What is the total amount due?
#   [0.9993]  answer: '$1,500.00'  span=(97,98)


config.json:   0%|          | 0.00/893 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/511M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/205 [00:00<?, ?it/s]

LayoutLMForQuestionAnswering LOAD REPORT from: impira/layoutlm-invoices
Key                              | Status     |  | 
---------------------------------+------------+--+-
token_classifier_head.weight     | UNEXPECTED |  | 
token_classifier_head.bias       | UNEXPECTED |  | 
layoutlm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/315 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

── Invoice-specialised model (impira/layoutlm-invoices) ──
  Q: What is the vendor name?
  [0.5541]  answer: 'John Smith'  span=(17,18)
  Q: What is the PO number?
  [0.0000]  answer: 'us-001 John Smith John Smith INVOICE DATE'  span=(16,22)
  Q: What is the subtotal?
  [0.9999]  answer: '145.00'  span=(69,69)
  Q: What is the tax amount?
  [1.0000]  answer: '9.06'  span=(73,73)
  Q: What is the total amount due?
  [0.9997]  answer: '$154.06'  span=(75,75)



In [16]:
# ── 6. Top-k answers — get multiple candidate answers ────────────────────────
#    top_k=N returns the N most likely answers ranked by confidence score
#    Useful when: the answer is ambiguous, or you want a confidence distribution
# ─────────────────────────────────────────────────────────────────────────────
pipe = pipeline(
    "document-question-answering",
    model="impira/layoutlm-document-qa",
)

print("── Top-k answers (top_k=3) ──")
img = load_image(CONTRACT)
result = pipe(image=img, question="What is the highest value?", top_k=3)
print(f"Q: What is the highest value?  (top_k=3, {len(result)} candidates)")
print_answers(result)
print()

# Expected output:
#   Q: What is the highest value?  (top_k=3, 3 candidates)
#   [0.7823]  answer: '3,848'   span=(22,22)
#   [0.1204]  answer: '12,540'  span=(18,18)
#   [0.0621]  answer: '1,950'   span=(30,30)



Loading weights:   0%|          | 0/205 [00:00<?, ?it/s]

LayoutLMForQuestionAnswering LOAD REPORT from: impira/layoutlm-document-qa
Key                              | Status     |  | 
---------------------------------+------------+--+-
layoutlm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


── Top-k answers (top_k=3) ──
Q: What is the highest value?  (top_k=3, 3 candidates)
  [0.2437]  answer: '$1,000,000,000'  span=(97,97)
  [0.2096]  answer: '$100,000,000,000.'  span=(171,171)
  [0.0003]  answer: '$100,000,000,000.'  span=(171,171)



In [18]:
# ── 7. Batch processing — list of (image, question) pairs in one call ─────────
#    Pass a list of dicts; returns a list of result lists
#    Best for: processing many documents, production pipelines
# ─────────────────────────────────────────────────────────────────────────────
pipe = pipeline(
    "document-question-answering",
    model="impira/layoutlm-document-qa",
)

batch = [
    {"image": load_image(INVOICE), "question": "What is the total amount?"},
    {"image": load_image(CONTRACT),    "question": "What is the applicant name?"},
]

print("── Batch document-qa ──")
batch_results = pipe(batch)

labels = ["invoice", "receipt", "form"]
for label, result in zip(labels, batch_results):
    top = result if isinstance(result, dict) else max(result, key=lambda x: x.get("score") or 0)
    score_str = f"{top['score']:.4f}" if top.get("score") else "N/A"
    print(f"  {label:10s}  →  [{score_str}]  {top['answer']!r}")
print()

# Expected output:
#   invoice    →  [0.9988]  '$1,500.00'
#   receipt    →  [0.9741]  '$24.17'
#   form       →  [0.8832]  'John Smith'

Loading weights:   0%|          | 0/205 [00:00<?, ?it/s]

LayoutLMForQuestionAnswering LOAD REPORT from: impira/layoutlm-document-qa
Key                              | Status     |  | 
---------------------------------+------------+--+-
layoutlm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


── Batch document-qa ──
  invoice     →  [0.9482]  '$154.06'
  receipt     →  [0.9952]  'UNICORN INDUSTRIES'



In [19]:
# ── 8. Word boxes (word + box coords) — skip internal OCR ────────────────────
#    If you already ran your own OCR (e.g. Tesseract, AWS Textract, Google Vision)
#    pass word tokens + bounding boxes directly to skip the internal OCR step
#    boxes are in (x0, y0, x1, y1) normalised to 0–1000 scale
# ─────────────────────────────────────────────────────────────────────────────
pipe = pipeline(
    "document-question-answering",
    model="impira/layoutlm-document-qa",
)

# Simulate OCR output — your real data would come from Tesseract / Textract / etc.
words = ["INVOICE", "NO:", "US-001", "DATE:", "11/02/2019", "TOTAL:", "$1,500.00"]
boxes = [
    [57,  10, 240, 35],   # INVOICE
    [57,  40, 100, 58],   # NO:
    [110, 40, 230, 58],   # US-001
    [57,  65, 102, 83],   # DATE:
    [112, 65, 252, 83],   # 11/02/2019
    [57,  90, 118, 108],  # TOTAL:
    [120, 90, 240, 108],  # $1,500.00
]

print("── Passing pre-computed word boxes (skip internal OCR) ──")
img = load_image(INVOICE)
result = pipe(
    image=img,
    question="What is the total?",
    word_boxes=list(zip(words, boxes)),
)
print("Q: What is the total?")
print_answers(result)
print()

# Expected output:
#   [0.9994]  answer: '$1,500.00'  span=(6,6)


Loading weights:   0%|          | 0/205 [00:00<?, ?it/s]

LayoutLMForQuestionAnswering LOAD REPORT from: impira/layoutlm-document-qa
Key                              | Status     |  | 
---------------------------------+------------+--+-
layoutlm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


── Passing pre-computed word boxes (skip internal OCR) ──
Q: What is the total?
  [0.9910]  answer: '$1,500.00'  span=(6,6)



In [21]:
# ── 9. Handle_impossible_answer — model can abstain ──────────────────────────
#    When the answer is not present in the document the model can return
#    answer="" with a very low score instead of hallucinating a span
#    Useful for: validation pipelines, QA over heterogeneous document sets
# ─────────────────────────────────────────────────────────────────────────────
pipe = pipeline(
    "document-question-answering",
    model="impira/layoutlm-document-qa",
)

print("── handle_impossible_answer=True ──")
img = load_image(CONTRACT)
result = pipe(
    image=img,
    question="What is the patient blood type?",   # not in a receipt
    handle_impossible_answer=True,
)
print("Q: What is the patient blood type?")
print_answers(result)
print()

# Expected output:
#   [0.0031]  answer: ''   ← empty string = model says "not found"

Loading weights:   0%|          | 0/205 [00:00<?, ?it/s]

LayoutLMForQuestionAnswering LOAD REPORT from: impira/layoutlm-document-qa
Key                              | Status     |  | 
---------------------------------+------------+--+-
layoutlm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


── handle_impossible_answer=True ──
Q: What is the patient blood type?
  [0.9863]  answer: ''  span=(0,0)

